# Labo 35 · Carnet Colab — PyTorch Geometric sur Cora

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=False)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATConv
appareil = "cuda" if torch.cuda.is_available() else "cpu"
cora = Planetoid("donnees", name="Cora")
data = cora[0].to(appareil)
print(data)
print("nœuds :", data.num_nodes, "· arêtes :", data.edge_index.shape[1] // 2, "· caractéristiques :", cora.num_features, "· classes :", cora.num_classes)
print("étiquetés pour l'entraînement :", int(data.train_mask.sum()), "· test :", int(data.test_mask.sum()))

## 1. Une boucle commune

`entrainer(modele)` entraîne 200 pas avec Adam (lr = 0.01, `weight_decay=5e-4`, une pénalité L2) sur les nœuds de `train_mask` seulement, et retourne l'exactitude sur `test_mask`. Chaque modèle reçoit `(x, edge_index)`.

In [ ]:
def entrainer(modele, pas=200):
    modele = modele.to(appareil)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01, weight_decay=5e-4)
    for _ in range(pas):
        modele.train()
        logits = modele(data.x, data.edge_index)
        perte = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
        optimiseur.zero_grad(); perte.backward(); optimiseur.step()
    modele.eval()
    with torch.no_grad():
        pred = modele(data.x, data.edge_index).argmax(dim=1)
    return round((pred[data.test_mask] == data.y[data.test_mask]).float().mean().item(), 4)

## 2. Trois modèles

**À toi** : complète les trois classes. Le perceptron **ignore** `edge_index` : `Linear(1433, 16)`, ReLU, dropout de 0.5, `Linear(16, 7)`. Le GCN remplace les deux couches linéaires par `GCNConv`. Le GAT utilise `GATConv(1433, 8, heads=8)` (8 têtes d'attention, concaténées : 64 sorties), ELU, puis `GATConv(64, 7, heads=1)`. Entraîne les trois et compare.

In [ ]:
class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        # À toi
    def forward(self, x, edge_index):
        pass   # À toi

class GCN(nn.Module):
    def __init__(self):
        super().__init__()
        # À toi
    def forward(self, x, edge_index):
        pass   # À toi

class GAT(nn.Module):
    def __init__(self):
        super().__init__()
        # À toi
    def forward(self, x, edge_index):
        pass   # À toi

# À toi : torch.manual_seed(0), puis entraîner les trois modèles

## 3. Voir les représentations du GCN

**À toi** : entraîne un GCN, récupère la sortie de sa **première** couche pour tous les nœuds (16 dimensions), projette-la en 2 dimensions avec `sklearn.manifold.TSNE`, et trace les nœuds colorés selon leur domaine. Fais de même avec les 1433 caractéristiques brutes (avec une ACP vers 50 dimensions avant TSNE, pour aller plus vite).

In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : l'exactitude au test du perceptron, du GCN et du GAT, le nombre de paramètres du GCN, et une phrase sur ce que montrent les deux projections.